In [9]:
import os
import ast
import pandas as pd
import networkx as nx
from tqdm import tqdm
import matplotlib.pyplot as plt

tqdm.pandas()

df = pd.read_csv(os.path.join(os.getcwd(),'raw_cluster.csv'))
df.head()

,cluster,cookies
0,14fe0abd-4bad-4da7-99bd-52ab6d461117,AWSALB
1,859b0c25-4482-4735-be22-d59313bde51f,userId
2,4c78f7ea-6540-433f-a60c-fa36de84fd75,ug
3,1d2fcd4a-7dcc-4210-8104-0b1849f5dac9,b
4,1982a795-1521-425d-8e09-240222eea0ad,WPtcs2


# Basic Stats

In [12]:
G = nx.from_pandas_edgelist(df, source='cluster', target='cookies')
clusters = {n for n in G.nodes() if isinstance(n, str) and n.count("-") >= 4 and len(n) >= 32}
cookies = set(G.nodes()) - clusters
print(f"[PY.ECO.1.0] Number of nodes: {G.number_of_nodes()}, number of edges: {G.number_of_edges()}")
print(f"[PY.ECO.1.1] mean connectivity: {G.number_of_edges() / G.number_of_nodes()}")
print(f"[PY.ECO.1.2] Graph density: {nx.density(G)}")
print(f"[PY.ECO.1.3] Is bipartite: {nx.is_bipartite(G)}")
bad_edges = [(u, v) for u, v in G.edges() if (u in clusters and v in clusters) or (u in cookies and v in cookies)]
print(f"[PY.ECO.1.4] Bad edges: {len(bad_edges)}")

[PY.ECO.1.0] Number of nodes: 28292, number of edges: 72097
[PY.ECO.1.0.1] mean connectivity: 2.5483175455959284
[PY.ECO.1.0.2] Graph density: 0.00018015040441100904
[PY.ECO.1.0.3] Is bipartite: True
[PY.ECO.1.0.3] Bad edges: 191


# Degree Distribution

In [13]:
from collections import Counter
deg = dict(G.degree())
cookie_deg_hist = Counter(deg[n] for n in cookies)
cluster_deg_hist = Counter(deg[n] for n in clusters)

top_cookies = sorted(((n, deg[n]) for n in cookies), key=lambda x: x[1], reverse=True)[:10]
top_clusters = sorted(((n, deg[n]) for n in clusters), key=lambda x: x[1], reverse=True)[:10]

print(f"[PY.ECO.2.0] top_cookies: {top_cookies}")
print(f"[PY.ECO.2.1] top_clusters: {top_clusters}")
print(f"[PY.ECO.2.2] cookie_degree_hist_sample: {cookie_deg_hist.most_common(10)}")

[PY.ECO.2.0] top_cookies: [('_gcl_au', 5102), ('_ga', 4907), ('_abck', 2262), ('__smn_fid', 1671), ('AMCV', 1601), ('OptanonConsent', 1506), ('AWSALB', 1108), ('_vwo_uuid_v2', 1077), ('_fbp', 972), ('PHPSESSID', 860)]
[PY.ECO.2.0] top_clusters: [('e68a7d6e-1548-4c14-a396-b42810ab610a', 614), ('8e087d38-fdcc-4ce5-b619-7a69cfa53af6', 597), ('370cf565-d06f-4d8f-99fb-a13f7f095bd4', 523), ('5fe01900-a62e-496c-bec6-6c1742911653', 521), ('b6d04bd8-5ccb-4dfd-bf8c-1aee634835df', 518), ('b5b7b001-fe3b-4120-a478-a2b73f2cde1b', 489), ('4ddcd8f1-f712-4813-a0b2-0e55822f19b1', 389), ('3e6a0165-f621-47e6-ad5b-25c6ca52fd2c', 368), ('75152ca5-4395-4f58-89fc-54d289ccca12', 330), ('97c42ab2-7227-4ba2-8d2e-9439f4b7d143', 328)]
[PY.ECO.2.0] cookie_degree_hist_sample: [(1, 1351), (2, 397), (3, 162), (4, 103), (5, 92), (6, 75), (7, 64), (8, 60), (9, 55), (10, 52)]


# Connected Components

In [14]:
cc = list(nx.connected_components(G))
cc_sorted = sorted(cc, key=len, reverse=True)
print(f"[PY.ECO.3.0] num_components: {len(cc_sorted)}, largest_component_size: {len(cc_sorted[0])}")

[PY.ECO.3.0] num_components: 162, largest_component_size: 25901


## Cookie Projection

In [15]:
largest = G.subgraph(cc_sorted[0]).copy()
clusters_l = clusters.intersection(largest.nodes())
cookies_l = cookies.intersection(largest.nodes())

cookie_proj = nx.bipartite.weighted_projected_graph(largest, cookies_l)
print(f"[PY.EC0.4.0] cookie_proj nodes/edges: {cookie_proj.number_of_nodes()}, {cookie_proj.number_of_edges()}")

[PY.EC0.4.0] cookie_proj nodes/edges: 3181, 625486


# Centrality

In [16]:
bet = nx.betweenness_centrality(G, k=2000, seed=42, normalized=True)
top_bet = sorted(bet.items(), key=lambda x: x[1], reverse=True)[:10]
print(f"[PY.EC0.5.0] top_betweenness_approx: {top_bet}")

# HITS needs a directed graph; this is a common trick for undirected graphs.
DG = G.to_directed()
hubs, auth = nx.hits(DG, max_iter=200, normalized=True)
top_auth = sorted(auth.items(), key=lambda x: x[1], reverse=True)[:10]
print(f"[PY.EC0.5.1] top_authorities: {top_auth}")

[PY.EC0.5.0] top_betweenness_approx: [('_ga', 0.21521253235500964), ('_gcl_au', 0.21014276644886995), ('_abck', 0.10112494329832022), ('AMCV', 0.05715297268590342), ('OptanonConsent', 0.04587563040402568), ('AWSALB', 0.03691551570442047), ('_vwo_uuid_v2', 0.035132115100319924), ('XSRF-TOKEN', 0.02827985639704172), ('PHPSESSID', 0.027373412164643098), ('_ym_uid', 0.026808264869032913)]
[PY.EC0.5.1] top_authorities: [('_gcl_au', 0.003716185586140287), ('_ga', 0.0035689442739455066), ('e68a7d6e-1548-4c14-a396-b42810ab610a', 0.0013469625672718367), ('8e087d38-fdcc-4ce5-b619-7a69cfa53af6', 0.0013043927191332821), ('b6d04bd8-5ccb-4dfd-bf8c-1aee634835df', 0.0012922931324607116), ('5fe01900-a62e-496c-bec6-6c1742911653', 0.0012749942488141071), ('OptanonConsent', 0.0012573711629473563), ('370cf565-d06f-4d8f-99fb-a13f7f095bd4', 0.0012573273008915488), ('b5b7b001-fe3b-4120-a478-a2b73f2cde1b', 0.0012271394701752743), ('_fbp', 0.0011799561872997263)]


# Community detection (Louvain on cookie projection)

In [17]:
from networkx.algorithms.community import louvain_communities

# Run Louvain on the projected cookie graph (usually more meaningful than on bipartite directly)
communities = louvain_communities(cookie_proj, weight="weight", seed=42)
communities = sorted(communities, key=len, reverse=True)
print(f"[PY.EC0.6.0] num_communities: {len(communities)}, largest_community_size: {len(communities[0])}")

[PY.EC0.6.0] num_communities: 20, largest_community_size: 1325
